# Flappy Bird RL - GPU Training

Train a Double DQN agent to play Flappy Bird using GPU acceleration.

**Setup:** Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Clone repo and install deps
!git clone https://github.com/tanmaysh17/flappy-bird.git
%cd flappy-bird
!pip install -q torch numpy

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Train headless - 5000 episodes on GPU
import os, time
import numpy as np
from flappy_rl.game import FlappyBirdEnv
from flappy_rl.agent import DQNAgent

env = FlappyBirdEnv()
agent = DQNAgent(device='cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {agent.device}")

os.makedirs('checkpoints', exist_ok=True)

NUM_EPISODES = 5000
scores = []
t0 = time.time()

for ep in range(NUM_EPISODES):
    state = env.reset()
    done = False
    while not done:
        action = agent.select_action(state, training=True)
        ns, r, done = env.step(action)
        agent.store_transition(state, action, r, ns, done)
        agent.train_step()
        state = ns
    scores.append(env.score)

    if ep % 100 == 0:
        avg = np.mean(scores[-100:])
        elapsed = time.time() - t0
        print(f'Ep {ep:5d} | Score: {env.score:3d} | Avg100: {avg:.1f} | '
              f'Eps: {agent.epsilon:.3f} | {elapsed:.0f}s')

    if ep > 0 and ep % 1000 == 0:
        agent.save(f'checkpoints/agent_ep{ep}.pt')

agent.save('checkpoints/agent_final.pt')
elapsed = time.time() - t0
print(f'\nDone in {elapsed:.0f}s | Final avg100: {np.mean(scores[-100:]):.1f} | Max: {max(scores)}')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(scores, alpha=0.3, color='cyan', linewidth=0.5)
if len(scores) >= 50:
    avg = np.convolve(scores, np.ones(50)/50, mode='valid')
    axes[0].plot(range(49, 49+len(avg)), avg, color='orange', linewidth=2)
axes[0].set_title('Score per Episode')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Score')
axes[0].grid(alpha=0.3)

# Score distribution
axes[1].hist(scores[-500:], bins=30, color='cyan', alpha=0.7, edgecolor='white')
axes[1].set_title('Score Distribution (last 500 eps)')
axes[1].set_xlabel('Score')
axes[1].axvline(np.mean(scores[-500:]), color='orange', linewidth=2, label=f'Mean: {np.mean(scores[-500:]):.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_results.png', dpi=150)
plt.show()
print(f'Best: {max(scores)} | Avg(last 100): {np.mean(scores[-100:]):.1f}')

In [ ]:
# Download the trained model
from google.colab import files
files.download('checkpoints/agent_final.pt')